In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)



In [17]:
X_train_np = np.load("X_train_np.npy")
X_test_np  = np.load("X_test_np.npy")
y_train_np = np.load("y_train_np.npy")
y_test_np  = np.load("y_test_np.npy")


In [18]:
class TripletDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.class_indices = {
            0: np.where(y == 0)[0],
            1: np.where(y == 1)[0]
        }

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        anchor = self.X[idx]
        anchor_label = self.y[idx]

        positive_idx = np.random.choice(self.class_indices[anchor_label])
        negative_label = 1 - anchor_label
        negative_idx = np.random.choice(self.class_indices[negative_label])

        positive = self.X[positive_idx]
        negative = self.X[negative_idx]

        return (
            torch.tensor(anchor),
            torch.tensor(positive),
            torch.tensor(negative)
        )


In [19]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, embedding_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), p=2, dim=1)


In [20]:
dataset = TripletDataset(X_np, y_np)
loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    drop_last=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Encoder(input_dim=X_np.shape[1]).to(device)
criterion = nn.TripletMarginLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [21]:
epochs = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for anchor, positive, negative in loader:
        anchor = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        optimizer.zero_grad()

        emb_anchor = model(anchor)
        emb_positive = model(positive)
        emb_negative = model(negative)

        loss = criterion(emb_anchor, emb_positive, emb_negative)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(loader):.4f}")


Epoch 1/15 - Loss: 0.0954
Epoch 2/15 - Loss: 0.0161
Epoch 3/15 - Loss: 0.0096
Epoch 4/15 - Loss: 0.0077
Epoch 5/15 - Loss: 0.0072
Epoch 6/15 - Loss: 0.0051
Epoch 7/15 - Loss: 0.0061
Epoch 8/15 - Loss: 0.0070
Epoch 9/15 - Loss: 0.0026
Epoch 10/15 - Loss: 0.0026
Epoch 11/15 - Loss: 0.0015
Epoch 12/15 - Loss: 0.0018
Epoch 13/15 - Loss: 0.0014
Epoch 14/15 - Loss: 0.0004
Epoch 15/15 - Loss: 0.0009


In [23]:
torch.save(encoder.state_dict(), "encoder_contrastivo_weights.pth")


